In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

c:\python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2829.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.getenv("GOOGLE_API_KEY"))

In [7]:
# response = llm.invoke("what 2 + 2 ?") 

In [8]:
# response.content

In [9]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [10]:
store={}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [11]:
config = {"configurable": {"session_id": "firstchat"}}
model_with_memory=RunnableWithMessageHistory(llm,get_session_history)

In [23]:
model_with_memory.invoke(("Hi! I'm Boss"),config=config).content

'Hi Boss! Nice to meet you.\n\nHow can I help you today?'

In [24]:
model_with_memory.invoke(("tell me what is my name?"),config=config).content

'Based on what you told me earlier, your name is **Boss**.'

In [12]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [13]:
if os.path.exists("../DATA"):
    print("Directory exists")
else:
    print("Directory does not exist")


Directory exists


In [14]:
### Reading the txt files from source directory

from langchain_community.document_loaders import PyPDFLoader

loader = DirectoryLoader('../DATA', glob="*.pdf", loader_cls=PyPDFLoader)
docs = loader.load()


In [15]:
### Creating Chunks using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

In [58]:
new_docs[7].page_content

'watching clouded sky. Do you remember, you used to perceive a particular shape (like, a football player, an\nelephant, map of a country or a state etc.) in the cloud, but your friend had witnessed some other shape?\nSituation-three: Remember, one crucial football/cricket match you were playing sometime back. There was\na garden close to playground. Do you remember, you could realize aggressive smell of manure in the garden\nonly when the match was over, but not during the tense moment of the match?'

In [59]:
doc_strings

['UNIT 6  PERCEPTION\nStructure\n6.0 Objectives\n6.1 Introduction\n6.2 Concept of Perception\n6.3 Process of Perception\n6.4 Factors Influencing Perception\n6.5 Barriers to Accurate Perception\n6.6 Theory of Attribution\n6.7 Managerial Uses of Perception\n6.8 Developing Perceptual Skills\n6.9 Let Us Sum Up\n6.10   Key Words\n6.11 Terminal Questions\n6.0 OBJECTIVES\nAfter studying this unit, you should be able to:\n• define the term Perception;\n• analyse  the determinants of Perception;',
 '• define the term Perception;\n• analyse  the determinants of Perception;\n• describe process of Perception;\n• identify uses of Perception in the field of human interaction;\n• explain the reasons for biases in Perception; and\n• identify ways to develop sound perceptual skills.\n1.1 INTRODUCTION\nIndividuals are exposed to varieties of  stimuli of the environment. They process these stimuli and interpret',
 'them.  The process of receiving information and making sense is known as perception.  It r

In [16]:
db = Chroma.from_documents(new_docs, embedding, collection_name="my_collection")

In [17]:
retriever = db.as_retriever(search_kwargs={"k": 4})

In [18]:

template = """You are a personal Assisteant.
Answer the question based only on the following context:
    {context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

In [19]:
rag_chain_memory = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | RunnableLambda(lambda x: x.to_string()) # to convert the retrieved context into string format -- why we did this? -> because the retrieved context is in list format and we need to convert it into string format to pass it to the model
    | model_with_memory
    | StrOutputParser()
)

In [20]:
question ="what is llama3? can you highlight 3 important points?"
# print(retrieval_chain.invoke(question))

In [21]:
def get_response(question: str) -> str:
    return retrieval_chain.invoke(question)

In [22]:
def get_response_memory(query:str) -> str:
    return rag_chain_memory.invoke(query , config={"configurable": {"session_id": "firstchat"}})

In [ ]:
# get_response("Hi i am Boss. My question is: What are Managerial uses of perception?")

'Perception is an important concept for managers. Some important managerial activities where the concept of perception can be applied include:\n\n*   advertising\n*   maintaining safety\n*   managing impression\n*   building corporate image\n*   managing performance\n*   evaluating performance\n*   judging employee’s loyalty\n*   self assessment'

In [ ]:
# get_response_memory("Hi i am Boss. My question is: What are Managerial uses of perception?")

'Perception is an important concept for managers. Based on the provided context, some important managerial activities where the concept of perception can be applied include:\n\n*   Advertising\n*   Maintaining safety\n*   Managing impression\n*   Building corporate image\n*   Managing performance\n*   Evaluating performance\n*   Judging employee’s loyalty\n*   Self assessment'

In [ ]:
# get_response_memory("Do you know my name?")

'Based on the provided context, there is no information that states I know your name.'

In [97]:
store

{'secondchat': InMemoryChatMessageHistory(messages=[HumanMessage(content="You are a personal Assisteant.\nAnswer the question based only on the following context:\n    [Document(metadata={'total_pages': 11, 'keywords': '', 'moddate': '2006-01-18T12:43:14+05:30', 'title': 'unit6', 'subject': 'unit6', 'creator': 'Adobe PageMaker 7.0', 'page_label': '1', 'creationdate': '2006-01-18T12:43:12+05:30', 'source': '..\\\\DATA\\\\Perception.pdf', 'author': 'Administrator', 'page': 0, 'producer': 'Acrobat Distiller 5.0.5 (Windows)'}, page_content='6.7 Managerial Uses of Perception'), Document(metadata={'title': 'unit6', 'author': 'Administrator', 'moddate': '2006-01-18T12:43:14+05:30', 'source': '..\\\\DATA\\\\Perception.pdf', 'keywords': '', 'creationdate': '2006-01-18T12:43:12+05:30', 'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'total_pages': 11, 'subject': 'unit6', 'page': 6, 'creator': 'Adobe PageMaker 7.0', 'page_label': '7'}, page_content='6.7 MANAGERIAL USES OF PERCEPTION'), Document(

## Tools and Agent


#### Wikipedia API Wrapper

In [23]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [24]:
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=200)

In [25]:
tool = WikipediaQueryRun(api_wrapper= api_wrapper)

In [26]:
tool.name

'wikipedia'

In [106]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [107]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [116]:
tool.run({"query": "langchain"})

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain'

#### Youtube Search API Wrapper

In [27]:
from langchain_community.tools import YouTubeSearchTool
tool1 = YouTubeSearchTool()

In [81]:
tool1.run("carryminati")

"['https://www.youtube.com/watch?v=lWBj9z2xDDs&pp=ygULY2FycnltaW5hdGk%3D', 'https://www.youtube.com/watch?v=abhuAYtmk58&pp=ygULY2FycnltaW5hdGk%3D']"

#### Web Search 

In [29]:
from langchain_community.tools.tavily_search import TavilySearchResults
tool3 = TavilySearchResults()

C:\Users\Manas tiwari\AppData\Local\Temp\ipykernel_43728\1394887549.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool3 = TavilySearchResults()


In [ ]:
# tool3.invoke({"query": "What happened in the latest burning man floods"})

[{'title': "How One Man Escaped From Burning Man 2023's Floods",
  'url': 'https://www.outsideonline.com/outdoor-adventure/exploration-survival/burning-man-2023-flooding-rain/',
  'content': 'A car that burst into flames while trying to drive out in the mud. Neighbors were able to put it out with fire extinguishers. [...] were a 15-minute trudge away, and they were almost certainly overflowing. So, I improvised. Grateful for once that I didn’t have a tent-mate, I popped a squat and crapped into a plastic CVS bag, tied it up real well, and tossed it into my garbage. Just like that, Burning Man 2023 no longer felt like just some “counter-culture festival” anymore. We began to wonder if FEMA or the National Guard was going to show up. [...] That night, it started raining again, so everybody retreated to their tents. Sunday morning, I woke up to the sounds of yelling and car engines. Despite the shelter-in-place order, some people were making a break for it, and others were pissed. An SUV 

#### Custom Tools

In [34]:
from langchain_core.tools import tool

In [61]:
@tool(
    "GET_STRING_LENGTH",
    description="Get the length of a string",
    args_schema={"s": str}
)
def get_string_length(s: str) -> int:
    return len(s)

In [69]:
print(get_string_length.name)
print(get_string_length.description)
print(get_string_length.args_schema)

GET_STRING_LENGTH
Get the length of a string
{'s': <class 'str'>}


In [49]:
res = get_string_length.invoke("Hello World")
print(res)

11


In [50]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

In [54]:
add_numbers.invoke({"a": 5, "b": 10})

15

In [68]:
print(add_numbers.name)
print(add_numbers.description)
print(add_numbers.args)

add_numbers
Add two numbers together.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### Concept of Agents 

#### ReAct Agent => Think -> Action -> Observation -> repeat(if needed)...
#### React Agent is a type of agent that uses a loop of thinking, acting, and observing to interact with its environment. It can be used in various applications, such as chatbots, virtual assistants, and decision-making systems. The agent thinks about the current situation, takes an action based on that thought, observes the outcome of the action, and then repeats the process if necessary. This allows the agent to learn and adapt over time based on its interactions with the environment.

### Why use ReAct?

*Reduces Hallucinations*: Because the model "observes" real data before answering, it is much less likely to make things up.

*Traceability*: You can see the agent's "Thought" process in the logs, which makes it much easier to debug why an agent went off-track.

*Tool-First*: It is the best pattern for tasks that require real-time data, like your "Pulse Chat" or "WatchNest" projects.



In [ ]:
# from langchain_community.agent_toolkits.load_tools import load_tools

In [77]:
# from langchain.agents import AgentType, initialize_agent

ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (c:\python313\Lib\site-packages\langchain\agents\__init__.py)

In [79]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    # Point to the OpenAI-compatible "door"
    base_url="http://localhost:11434/v1", 
    
    # Required but ignored by Ollama
    api_key="ollama", 
    
    # The exact name you see in 'ollama list'
    model="gemma3:4b", 
    
    temperature=0
)

In [80]:
# Test it
print(llm.invoke("What is the time complexity of Merge Sort?").content)

The time complexity of Merge Sort is **O(n log n)**, where 'n' is the number of elements being sorted.  Let's break down why:

* **Divide and Conquer:** Merge Sort is a classic example of a "divide and conquer" algorithm. It recursively breaks down the problem into smaller subproblems until each subproblem contains only one element (which is trivially sorted).

* **Merge Step:** The key to the efficiency is the "merge" step.  Merging two sorted lists of size 'n/2' into a single sorted list takes O(n) time.  This is because you're comparing elements from both lists and placing them in the correct order.

* **Recursion:**
    * The first level of division takes O(log n) time (because you're dividing the problem in half at each step).
    * The merging steps at each level take O(n) time.
    * Since there are log n levels of recursion, the total time is O(n log n).

**Here's a more detailed breakdown:**

1. **Divide:** The original array of size 'n' is divided into two halves of size n/2.

In [ ]:
# tools = [tool3]

In [89]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [90]:
loader = WebBaseLoader("https://docs.smith.langchain.com/overview")
docs = loader.load()

In [92]:
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

In [93]:
vector = FAISS.from_documents(documents, embedding)
retriever = vector.as_retriever(search_kwargs={"k": 5})

In [97]:
retriever.invoke("What is LangSmith?")[0].page_content

'LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.'

#### Creating retriver tool **(Legacy code, not working)**

In [ ]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",
)

In [ ]:
tools = [search, retriever_tool]
from langchain.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm, tools, prompt)
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "hi! what is a langsmith?"})

## ReAct Agent (Reasoning + Acting)

### Definition

The **ReAct** framework combines **reasoning** and **acting** in a single loop to handle tasks. The agent uses natural language reasoning (thinking through steps) and task actions (performing tasks like calculations or data retrieval).

The agent uses a combination of reasoning steps to guide actions in real-time, using feedback from those actions to further inform the next step in reasoning.

### How it Works

1. **Step 1:** The agent receives a question or task.
2. **Step 2:** It reasons aloud (in natural language) about how to solve it.
3. **Step 3:** Based on its reasoning, it takes actions (e.g., searching a database, calculating something).
4. **Step 4:** The results of these actions are integrated into its reasoning and may trigger further actions.
5. **Step 5:** It repeats the process until it arrives at a solution.

### Key Points

- **Combines thinking and doing** (reasoning and actions)
- **Performs iterative steps**, updating its process based on action results
- **Typically handles complex decision-making** scenarios
- **Reduces hallucinations** by observing real data before answering

### Example

**Original Task:**  
_"Calculate the total number of apples in a basket if there are 4 baskets and 7 apples in each."_

| Step | Description |
|------|-------------|
| **Thinking** | "I need to multiply 4 by 7 to get the total number of apples." |
| **Action** | Perform the multiplication: 4 × 7 |
| **Result** | 28 apples |
| **Conclusion** | "The task is complete." |


## Self-Ask with Search Agent

### Definition

This approach allows the AI to **ask itself follow-up questions** when it doesn't immediately know the answer. It performs a **recursive questioning process** to break down complex queries into smaller, more manageable ones.

Once the AI generates these follow-up questions, it performs an **external search** (e.g., through Google or an internal database) to gather the necessary information before formulating a response.

### How It Works

1. **Step 1:** Receive a complex question.
2. **Step 2:** The agent identifies sub-questions or follow-up questions.
3. **Step 3:** It performs a search or fetches answers to these questions from external resources (like a web search).
4. **Step 4:** The answers are aggregated to provide a complete response.

### Key Points

- **Emphasizes question decomposition** — breaking down complex queries into simpler parts
- **Relies on external search** for sub-questions and supporting information
- **Useful for open-ended or broad questions** where the answer is not immediately available
- **Iterative refinement** — progressively builds understanding through multiple searches

### Example

**Original Question:**  
_"How many moons does Jupiter have?"_

| Step | Action | Description |
|------|--------|-------------|
| **Question 1** | Ask | "What is Jupiter?" |
| **Search 1** | Search | Find information about Jupiter |
| **Question 2** | Ask | "What are moons?" |
| **Search 2** | Search | Find information about moons |
| **Aggregation** | Combine | Gather all relevant information |
| **Final Answer** | Answer | "Jupiter has 79 known moons." |


In [107]:
# Legacy code for retriever tool, not working
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\python313\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
google_search = GoogleSerperAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=google_search.run,
        description="useful for when you need to ask with search",
        verbose=True
    )
]

In [110]:
template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}'''

In [ ]:
prompt = PromptTemplate.from_template(template)
search_agent = create_react_agent(llm,tools,prompt)

In [ ]:
agent_executor = AgentExecutor(
    agent=search_agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
)
agent_executor.invoke({"input": "Where is the hometown of the 2007 US PGA championship winner and his score?"})

### ReAct Agent with Custom tools

In [ ]:
from langchain.tools import tool
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent

In [113]:
# Custom tool for the Agent 
@tool
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}
  
  return fake_employees.get(name,"Employee not found")

@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [ ]:
# Saved React Prompt in langchain hub, we could manually type the prompt as well.
prompt = hub.pull("hwchase17/react")
print(prompt.template)

In [ ]:
tools = [get_employee_salary, get_employee_id]
agent = create_react_agent(llm,tools,prompt)
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor.invoke({"input":"What is the Salary of Evan?"})

#### Output of the agent

> Entering new AgentExecutor chain...
Thought: I need to find Evan's employee ID first.
Action: get_employee_id
Action Input: EvanE005Thought: Now that I have Evan's employee ID, I can find his salary.
Action: get_employee_salary
Action Input: E00545000I now know the final answer
Final Answer: 45000 


> Finished chain.
{'input': 'What is the Salary of Evan?', 'output': '45000'}